In [1]:
import redis.asyncio as redis  
import asyncio
from datetime import datetime
import nest_asyncio

nest_asyncio.apply()

from alpaca.data.live.stock import StockDataStream
import os 

stock_stream = StockDataStream(os.environ['API_KEY'], os.environ['SECRET_KEY'])
tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

async def push_ohlc_data(bar):
    bar = {k: v for k, v in bar}
    bar['timestamp'] = bar['timestamp'].isoformat()
    # Add the new OHLC tick to the Redis Stream
    await redis_client.xadd(f"alpaca_{bar['symbol']}", bar)

    # Trim the stream to keep only the last 30 ticks
    await redis_client.xtrim(f"ohlc_stream:{bar['symbol']}", maxlen=100)
    print(f"Pushed OHLC Tick: {bar}")


In [ ]:
stock_stream.subscribe_bars(push_ohlc_data, *tickers)
stock_stream.run()

Pushed OHLC Tick: {'symbol': 'CSCO', 'timestamp': '2025-03-19T14:50:00+00:00', 'open': 60.695, 'high': 60.695, 'low': 60.675, 'close': 60.675, 'volume': 500.0, 'trade_count': 7.0, 'vwap': 60.6825}
Pushed OHLC Tick: {'symbol': 'MSFT', 'timestamp': '2025-03-19T14:50:00+00:00', 'open': 386.38, 'high': 386.38, 'low': 386.2, 'close': 386.2, 'volume': 259.0, 'trade_count': 4.0, 'vwap': 386.29}
Pushed OHLC Tick: {'symbol': 'AMD', 'timestamp': '2025-03-19T14:50:00+00:00', 'open': 104.15, 'high': 104.22, 'low': 104.08, 'close': 104.22, 'volume': 1479.0, 'trade_count': 19.0, 'vwap': 104.165714}
Pushed OHLC Tick: {'symbol': 'NVDA', 'timestamp': '2025-03-19T14:50:00+00:00', 'open': 116.77, 'high': 116.795, 'low': 116.7, 'close': 116.795, 'volume': 5261.0, 'trade_count': 62.0, 'vwap': 116.744735}
Pushed OHLC Tick: {'symbol': 'TSLA', 'timestamp': '2025-03-19T14:50:00+00:00', 'open': 231.46, 'high': 231.955, 'low': 231.36, 'close': 231.955, 'volume': 989.0, 'trade_count': 15.0, 'vwap': 231.626111}
Pu